# SpaceGuard AI — Dataset Inventory: NASA SMAP / MSL Telemanom

> **Status:** Dataset Verification & Readability Audit  
> **Notice:** **ML MODEL TRAINING HAS NOT STARTED.**  
> This notebook performs a read-only inspection of the acquired raw NASA SMAP/MSL benchmark dataset. No preprocessing, feature engineering, or model training is performed.

## 1. Dataset Paths & Provenance Manifest

In [1]:
import os
import json
from pathlib import Path

# Locate raw dataset and manifest
base_dir = Path.cwd().resolve()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent

raw_dir = base_dir / "data" / "raw" / "nasa_smap_msl"
manifest_path = base_dir / "data" / "DATASET_MANIFEST.json"

print(f"Raw dataset directory : {raw_dir}")
print(f"Raw directory exists  : {raw_dir.exists()}")
print(f"Manifest file exists  : {manifest_path.exists()}")

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"\nDataset Name : {manifest['datasetName']}")
    print(f"Source       : {manifest['source']}")
    print(f"Total Files  : {manifest['totalFiles']}")
    print(f"Total Size   : {manifest['totalSizeMB']} MB")

Raw dataset directory : C:\Users\Aman\OneDrive\Desktop\SpaceGaurd\ml-service\data\raw\nasa_smap_msl
Raw directory exists  : True
Manifest file exists  : True

Dataset Name : NASA SMAP and MSL Telemetry Anomaly Benchmark
Source       : NASA Jet Propulsion Laboratory (JPL) / Hundman et al., KDD 2018
Total Files  : 7
Total Size   : 163.24 MB


## 2. Channel Overview & Label Manifest

Inspect the authoritative ground truth annotation definitions from `labeled_anomalies.csv`.

In [2]:
import pandas as pd

labels_file = raw_dir / "labeled_anomalies.csv"
df_labels = pd.read_csv(labels_file)

print(f"Total Channels Cataloged : {len(df_labels)}")
print("\nChannel breakdown by spacecraft:")
print(df_labels["spacecraft"].value_counts())

print("\nFirst 8 cataloged channels:")
df_labels.head(8)

Total Channels Cataloged : 82

Channel breakdown by spacecraft:
spacecraft
SMAP    55
MSL     27
Name: count, dtype: int64

First 8 cataloged channels:


,chan_id,spacecraft,anomaly_sequences,class,num_values
0,P-1,SMAP,"[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]",8505
1,S-1,SMAP,"[[5300, 5747]]",[point],7331
2,E-1,SMAP,"[[5000, 5030], [5610, 6086]]","[contextual, contextual]",8516
3,E-2,SMAP,"[[5598, 6995]]",[point],8532
4,E-3,SMAP,"[[5094, 8306]]",[point],8307
5,E-4,SMAP,"[[5450, 8261]]",[point],8354
6,E-5,SMAP,"[[5600, 5920]]",[point],8294
7,E-6,SMAP,"[[5610, 5675]]",[point],8300


## 3. Telemetry Array Dimensions & Shapes

Load the raw NumPy arrays for both MSL (Mars Science Laboratory) and SMAP (Soil Moisture Active Passive).

In [3]:
import numpy as np

# Load MSL arrays
msl_train = np.load(raw_dir / "MSL_train.npy")
msl_test = np.load(raw_dir / "MSL_test.npy")
msl_test_label = np.load(raw_dir / "MSL_test_label.npy")

# Load SMAP arrays
smap_train = np.load(raw_dir / "SMAP_train.npy")
smap_test = np.load(raw_dir / "SMAP_test.npy")
smap_test_label = np.load(raw_dir / "SMAP_test_label.npy")

print("=" * 60)
print("MSL TELEMETRY ARRAYS:")
print(f"  Train split shape       : {msl_train.shape} (dtype: {msl_train.dtype})")
print(f"  Test split shape        : {msl_test.shape} (dtype: {msl_test.dtype})")
print(f"  Test label mask shape   : {msl_test_label.shape} (dtype: {msl_test_label.dtype})")

print("\nSMAP TELEMETRY ARRAYS:")
print(f"  Train split shape       : {smap_train.shape} (dtype: {smap_train.dtype})")
print(f"  Test split shape        : {smap_test.shape} (dtype: {smap_test.dtype})")
print(f"  Test label mask shape   : {smap_test_label.shape} (dtype: {smap_test_label.dtype})")
print("=" * 60)

MSL TELEMETRY ARRAYS:
  Train split shape       : (58317, 55) (dtype: float64)
  Test split shape        : (73729, 55) (dtype: float64)
  Test label mask shape   : (73729,) (dtype: bool)

SMAP TELEMETRY ARRAYS:
  Train split shape       : (135183, 25) (dtype: float64)
  Test split shape        : (427617, 25) (dtype: float64)
  Test label mask shape   : (427617,) (dtype: bool)


## 4. Anomaly Label Statistics

Calculate exact positive anomaly rates and point vs contextual distributions.

In [4]:
msl_anomaly_count = int(msl_test_label.sum())
msl_total = len(msl_test_label)
msl_anomaly_pct = (msl_anomaly_count / msl_total) * 100

smap_anomaly_count = int(smap_test_label.sum())
smap_total = len(smap_test_label)
smap_anomaly_pct = (smap_anomaly_count / smap_total) * 100

print(f"MSL Test Anomaly Ratio  : {msl_anomaly_count} / {msl_total} points ({msl_anomaly_pct:.2f}%)")
print(f"SMAP Test Anomaly Ratio : {smap_anomaly_count} / {smap_total} points ({smap_anomaly_pct:.2f}%)")

MSL Test Anomaly Ratio  : 7766 / 73729 points (10.53%)
SMAP Test Anomaly Ratio : 54696 / 427617 points (12.79%)


## 5. Raw Telemetry Inspection (No Preprocessing)

Display first 5 timesteps across leading channels to observe raw data structure.

In [5]:
print("MSL Train - First 5 timesteps (first 5 channels):")
print(msl_train[:5, :5])

print("\nSMAP Train - First 5 timesteps (first 5 channels):")
print(smap_train[:5, :5])

MSL Train - First 5 timesteps (first 5 channels):
[[2.14664587 0.         0.         0.         0.        ]
 [2.14664587 0.         0.         0.         0.        ]
 [2.14664587 0.         0.         0.         0.        ]
 [2.15132605 0.         0.         0.         0.        ]
 [2.16380655 0.         0.         0.         0.        ]]

SMAP Train - First 5 timesteps (first 5 channels):
[[0.999 0.    0.    0.    0.   ]
 [0.999 0.    0.    0.    0.   ]
 [0.999 0.    0.    0.    0.   ]
 [0.999 0.    0.    0.    0.   ]
 [0.999 0.    0.    0.    0.   ]]


## 6. Missingness & NaN Check

Perform non-destructive check for NaN or infinite values.

In [6]:
msl_nan = np.isnan(msl_train).sum() + np.isnan(msl_test).sum()
smap_nan = np.isnan(smap_train).sum() + np.isnan(smap_test).sum()

print(f"MSL Missing / NaN count  : {msl_nan}")
print(f"SMAP Missing / NaN count : {smap_nan}")
if msl_nan == 0 and smap_nan == 0:
    print("\n[OK] Raw telemetry arrays contain complete, non-null numeric floats.")

MSL Missing / NaN count  : 0
SMAP Missing / NaN count : 0

[OK] Raw telemetry arrays contain complete, non-null numeric floats.


## 7. Dataset Boundaries & Summary

- Raw data is stored immutably in `data/raw/nasa_smap_msl/`.
- This dataset is segregated for ML methodology research and does NOT replace SpaceGuard production telemetry.
- **ML MODEL TRAINING HAS NOT STARTED.**